In [0]:
from pyspark.sql import functions as F

T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_locations = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_items = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_subscriber = spark.table("cpx_dataanalytics_gold.marketing.subscriber_profile_snapshot")
T_vouchTrans = spark.table("cpx_dataanalytics_gold.marketing.tbltransvoucher")
T_vouchDesc = spark.table("cpx_dataanalytics_gold.marketing.tblvouchertype")

In [0]:
#We use the snapshot date as the anchorpoint, so start date is 12months before the snapshot date.

snapshot_date = F.to_date(F.lit("20250801"), "yyyyMMdd")
start_date = F.add_months(snapshot_date, -12)
end_date = snapshot_date

T_master_trans = (T_trans
    .join(T_locations, T_trans.LocationId == T_locations.LocationID, "left")
    .join(T_items, T_trans.ItemSkey == T_items.itemskey, "left")
    .join(T_date.select("dateid", "datefull"), T_trans.VisitDateId == T_date.dateid, "left")
    .filter(
        (T_locations.LC_Location_Type_Description == "Cineplex Theatre")
        & (F.col("STOREFLAG") == "No")
        & (F.col("RecRoomFLAG") == "No")
        & (T_trans.CDE_ID.isNotNull())
        & (F.col("datefull").between(start_date, end_date))
        & ((F.col("TicketFLAG") == "Yes") | (F.col("ConcessionFLAG") == "Yes"))
    )
    .withColumnRenamed("datefull", "VisitDate")
)


In [0]:
# Define upgrade window (inclusive)
upgrade_start_date = "2025-07-01"
upgrade_end_date   = "2025-07-31"

snapshot_date_id = "20250731"

#Members that Upgraded to Annual
df_upgraded_members = (
    T_subscriber
    .filter(F.col("renewal_status") == "Active_Annnual_Upgrade")
    .groupBy("cde_id")
    .agg(
        F.min(
            F.to_date(
                F.col("snapshot_date_id").cast("string"),
                "yyyyMMdd"
            )
        ).alias("date_upgraded")
    )
    .filter(F.col("date_upgraded").between(upgrade_start_date, upgrade_end_date))
    .select("cde_id")
    .distinct()
)

#Members that Joined on the Activated_at_datetime
df_active_annual_snapshot = (
    T_subscriber.filter(F.col("snapshot_date_id") == snapshot_date_id)
    .filter(F.to_date(T_subscriber.activated_at_datetime).between(upgrade_start_date, upgrade_end_date))
    .filter((F.col("plan_id") == "annual1") & (F.col("renewal_status").like("%Active%")))
    .select('cde_id',T_subscriber.activated_at_datetime)
    .distinct()
)

#All instances of Activation
df_first_activation_new = (
    T_subscriber
    .groupBy("cde_id")
    .agg(
        F.min(F.to_date(T_subscriber.activated_at_datetime))
    )
    .withColumnRenamed("min(to_date(activated_at_datetime))","first_activation")
)

#Joining the 2 tables
df_details = (df_active_annual_snapshot
              .join(df_first_activation_new, on="cde_id", how="inner")
              )

#Returning Members
df_returning = (
    df_details
    .filter(~F.col("first_activation").between(upgrade_start_date, upgrade_end_date))
)

#New Members
df_new_signups = (
    df_details
    .filter(F.col("first_activation").between(upgrade_start_date, upgrade_end_date))
)

#List of all CDE_IDs
df_eligible_members = (
    df_upgraded_members
    .union(df_details.select("cde_id"))
    .distinct()
)

#Matching them with the Main Transaction Table
df_all_member_trans = df_eligible_members.join(T_master_trans, "CDE_ID", "inner")



In [0]:
overlap_upgraded_returning = (
    df_upgraded_members.select("cde_id")
    .intersect(df_returning.select("cde_id"))
)
print("Overlap upgraded vs returning:", overlap_upgraded_returning.count())


In [0]:
overlap_upgraded_returning.display()


In [0]:
display(df_details.count())

In [0]:
display(df_returning.count())

In [0]:
display(df_new_signups.count())

In [0]:
df_eligible_members.display()

In [0]:
df_upgraded_flag = (
    df_upgraded_members
    .withColumn("is_upgraded", F.lit(1))
)

df_returning_flag = (
    df_returning
    .select("cde_id")
    .withColumn("is_returning", F.lit(1))
)

df_new_flag = (
    df_new_signups
    .select("cde_id")
    .withColumn("is_new", F.lit(1))
)

df_classification = (
    df_upgraded_flag
    .join(df_returning_flag, "cde_id", "full")
    .join(df_new_flag, "cde_id", "full")
    .fillna(0)
)

df_final_segments = (
    df_classification
    .withColumn(
        "member_type",
        F.when(F.col("is_upgraded") == 1, "Upgraded")
         .when(F.col("is_new") == 1, "New")
         .when(F.col("is_returning") == 1, "Returning")
         .otherwise("Unclassified")
    )
)

df_distribution = (
    df_final_segments
    .groupBy("member_type")
    .agg(F.countDistinct("cde_id").alias("member_count"))
    .orderBy("member_type")
)

display(df_distribution)


In [0]:
df_visits = (
    df_all_member_trans
    .select("CDE_ID", "VisitDate")
    .dropDuplicates()
)

df_visit_counts = (
    df_visits
    .groupBy("CDE_ID")
    .agg(
        F.count("*").alias("total_visits_p12")
    )
)

df_member_visits = (
    df_eligible_members
    .join(df_visit_counts, "CDE_ID", "left")
    .fillna(0, subset=["total_visits_p12"])
)

df_visit_history = (
    df_member_visits
    .withColumn(
        "visit_bucket",
        F.when(F.col("total_visits_p12") == 0, "0")
         .when(F.col("total_visits_p12") == 1, "1")
         .when(F.col("total_visits_p12").between(2, 3), "2–3")
         .when(F.col("total_visits_p12").between(4, 6), "4–6")
         .when(F.col("total_visits_p12").between(7, 12), "7–12")
         .otherwise("13+")
    )
)

df_visit_category_counts = (
    df_visit_history
    .groupBy("visit_bucket")
    .agg(
        F.countDistinct("CDE_ID").alias("no_of_people")
    )
    .orderBy("visit_bucket")
)

df_visit_category_counts.display()


In [0]:
df_avg_visits = (
    df_visit_history
    .agg(
        F.avg("Total_Visits_P12").alias("Avg_Visits_P12")
    )
)

display(df_avg_visits)

In [0]:
df_voucher_trans = (
    T_vouchTrans.alias("vt")
    .join(
        T_vouchDesc.alias("vd"),
        F.col("vt.voucher_type_id") == F.col("vd.voucher_type_id"),
        how="left"
    )
    .filter(F.col("n_voucher_code").like("7124%"))
)
#7132 is for Nachos, 7124 is for Popcorn

display(df_voucher_trans.count())

In [0]:
V1 = df_voucher_trans.alias("V1")
V3 = T_trans.alias("V3")

df_voucher_trans_joined = (
    V1
    .withColumn(
        "session_date_id",
        F.date_format(F.col("session_dtm_realshow"), "yyyyMMdd")
    )
    .join(
        V3.select(
            "CDE_ID",
            "LocationId",
            "Transaction_Number",
            "VisitDateId"
        ),
        (V1.l_location_id == V3.LocationId) &
        (V1.trans_c_lgn_number == V3.Transaction_Number) &
        (F.col("session_date_id") == V3.VisitDateId),
        how="left"
    )
    .withColumn(
        "Transaction_Matched_Flag",
        F.when(F.col("Transaction_Number").isNotNull(), 1).otherwise(0)
    )
    .select("CDE_ID")
    .distinct()
)


In [0]:
from pyspark.sql import functions as F

# ----------------------------
# 1) Build transaction-level base (P12M window)
# ----------------------------
df_popcorn_base = (
    df_voucher_trans_joined
    .join(T_trans, "CDE_ID", how="left")
    .join(T_subscriber.filter(F.col("snapshot_date_id") == "20251001"), "CDE_ID", how="left")
    .join(
        T_items.select(F.col("itemskey").alias("ItemSkey"), "Item_Description"),
        on="ItemSkey",
        how="left"
    )
    .withColumn("Visit_Date", F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd"))
    .filter(F.col("Visit_Date").between("2024-08-01", "2025-08-30"))
)

# ----------------------------
# 2) One row per visit with net popcorn quantity (refund-safe via Net_Popcorn_Qty > 0)
#    Visit grain assumed: (CDE_ID, VisitDateId, LocationId, plan_id)
# ----------------------------
df_visit_popcorn = (
    df_popcorn_base
    .groupBy("CDE_ID", "VisitDateId", "LocationId", "plan_id")
    .agg(
        F.sum(
            F.when(F.col("Item_Description").rlike("(?i)popcorn"), F.col("Quantity")).otherwise(F.lit(0))
        ).alias("Net_Popcorn_Qty")
    )
    .withColumn("Popcorn_Visit_Flag", F.when(F.col("Net_Popcorn_Qty") > 0, F.lit(1)).otherwise(F.lit(0)))
)

# ----------------------------
# 3) Customer-level table (requested columns + visit-level popcorn quantity buckets)
# ----------------------------
df_popcorn_incidence_final = (
    df_visit_popcorn
    .groupBy("CDE_ID", "plan_id")
    .agg(
        F.count("*").alias("Total_Visits"),
        F.sum("Popcorn_Visit_Flag").alias("Popcorn_Visits"),

        # visit-level buckets for net popcorn quantity
        F.sum(F.when(F.col("Net_Popcorn_Qty") == 1, 1).otherwise(0)).alias("Bought_1_Popcorn"),
        F.sum(F.when(F.col("Net_Popcorn_Qty") == 2, 1).otherwise(0)).alias("Bought_2_Popcorns"),
        F.sum(F.when(F.col("Net_Popcorn_Qty") > 2, 1).otherwise(0)).alias("Bought_More_Than_2_Popcorns"),
    )
    .withColumn(
        "Popcorn_Incidence_Total",
        F.when(F.col("Total_Visits") > 0, F.col("Popcorn_Visits") / F.col("Total_Visits")).otherwise(F.lit(0.0))
    )
    .select(
        "CDE_ID",
        "Total_Visits",
        "Popcorn_Visits",
        "Popcorn_Incidence_Total",
        "Bought_1_Popcorn",
        "Bought_2_Popcorns",
        "Bought_More_Than_2_Popcorns",
        "plan_id"
    )
)

display(df_popcorn_incidence_final)


In [0]:
df_voucher_trans = (
    T_vouchTrans.alias("vt")
    .join(
        T_vouchDesc.alias("vd"),
        F.col("vt.voucher_type_id") == F.col("vd.voucher_type_id"),
        how="left"
    )
    .filter(F.col("n_voucher_code").like("7132%"))
)
#7132 is for Nachos, 7124 is for Popcorn

display(df_voucher_trans.count())

V1 = df_voucher_trans.alias("V1")
V3 = T_trans.alias("V3")

df_voucher_trans_joined = (
    V1
    .withColumn(
        "session_date_id",
        F.date_format(F.col("session_dtm_realshow"), "yyyyMMdd")
    )
    .join(
        V3.select(
            "CDE_ID",
            "LocationId",
            "Transaction_Number",
            "VisitDateId"
        ),
        (V1.l_location_id == V3.LocationId) &
        (V1.trans_c_lgn_number == V3.Transaction_Number) &
        (F.col("session_date_id") == V3.VisitDateId),
        how="left"
    )
    .withColumn(
        "Transaction_Matched_Flag",
        F.when(F.col("Transaction_Number").isNotNull(), 1).otherwise(0)
    )
    .select("CDE_ID")
    .distinct()
)


In [0]:
from pyspark.sql import functions as F

# ----------------------------
# 1) Build transaction-level base (P12M window)
# ----------------------------
df_Nachos_base = (
    df_voucher_trans_joined
    .join(T_trans, "CDE_ID", how="left")
    .join(T_subscriber.filter(F.col("snapshot_date_id") == "20251201"), "CDE_ID", how="left")
    .join(
        T_items.select(F.col("itemskey").alias("ItemSkey"), "Item_Description"),
        on="ItemSkey",
        how="left"
    )
    .withColumn("Visit_Date", F.to_date(F.col("VisitDateId").cast("string"), "yyyyMMdd"))
    .filter(F.col("Visit_Date").between("2024-11-01", "2025-11-30"))
)

# ----------------------------
# 2) One row per visit with net Nachos quantity (refund-safe via Net_Nachos_Qty > 0)
#    Visit grain assumed: (CDE_ID, VisitDateId, LocationId, plan_id)
# ----------------------------
df_visit_Nachos = (
    df_Nachos_base
    .groupBy("CDE_ID", "VisitDateId", "LocationId", "plan_id")
    .agg(
        F.sum(
            F.when(F.col("Item_Description").rlike("(?i)Nachos"), F.col("Quantity")).otherwise(F.lit(0))
        ).alias("Net_Nachos_Qty")
    )
    .withColumn("Nachos_Visit_Flag", F.when(F.col("Net_Nachos_Qty") > 0, F.lit(1)).otherwise(F.lit(0)))
)

# ----------------------------
# 3) Customer-level table (requested columns + visit-level Nachos quantity buckets)
# ----------------------------
df_Nachos_incidence_final = (
    df_visit_Nachos
    .groupBy("CDE_ID", "plan_id")
    .agg(
        F.count("*").alias("Total_Visits"),
        F.sum("Nachos_Visit_Flag").alias("Nachos_Visits"),

        # visit-level buckets for net Nachos quantity
        F.sum(F.when(F.col("Net_Nachos_Qty") == 1, 1).otherwise(0)).alias("Bought_1_Nachos"),
        F.sum(F.when(F.col("Net_Nachos_Qty") == 2, 1).otherwise(0)).alias("Bought_2_Nachos"),
        F.sum(F.when(F.col("Net_Nachos_Qty") > 2, 1).otherwise(0)).alias("Bought_More_Than_2_Nachos"),
    )
    .withColumn(
        "Nachos_Incidence_Total",
        F.when(F.col("Total_Visits") > 0, F.col("Nachos_Visits") / F.col("Total_Visits")).otherwise(F.lit(0.0))
    )
    .select(
        "CDE_ID",
        "Total_Visits",
        "Nachos_Visits",
        "Nachos_Incidence_Total",
        "Bought_1_Nachos",
        "Bought_2_Nachos",
        "Bought_More_Than_2_Nachos",
        "plan_id"
    )
)

display(df_Nachos_incidence_final)
